# Explore cross-validation results and final test evaluation for the biologically inspired model

## Data loading and preprocessing

In [1]:
import numpy as np
import pandas as pd

def subject_stats(df, subject_col='subjectID', fog_col='fog'):
    g = df.groupby(subject_col)[fog_col]
    stats = pd.DataFrame({
        'n_total': g.size(),
        'n_fog': g.sum()
    }).reset_index()
    stats['fog_rate'] = stats['n_fog'] / stats['n_total']
    return stats

def evaluate_split(stats, train_subj, val_subj, test_subj, target_sizes=(0.6, 0.2, 0.2), alpha=1.0, beta=0.5):
    all_n = stats['n_total'].sum()
    global_rate = stats['n_fog'].sum() / all_n

    def split_metrics(subj_list):
        x = stats[stats['subjectID'].isin(subj_list)]
        n = x['n_total'].sum()
        r = (x['n_fog'].sum() / n) if n > 0 else 0.0
        return n, r

    n_tr, r_tr = split_metrics(train_subj)
    n_va, r_va = split_metrics(val_subj)
    n_te, r_te = split_metrics(test_subj)

    rate_term = abs(r_tr - global_rate) + abs(r_va - global_rate) + abs(r_te - global_rate)

    size_tr, size_va, size_te = n_tr / all_n, n_va / all_n, n_te / all_n
    size_term = abs(size_tr - target_sizes[0]) + abs(size_va - target_sizes[1]) + abs(size_te - target_sizes[2])

    score = alpha * rate_term + beta * size_term
    return score, {
        'train_rate': r_tr, 'val_rate': r_va, 'test_rate': r_te,
        'global_rate': global_rate,
        'train_size': size_tr, 'val_size': size_va, 'test_size': size_te
    }

def random_subject_split_search(df, n_iter=20000, seed=42, ratios=(0.6, 0.2, 0.2)):
    rng = np.random.default_rng(seed)
    stats = subject_stats(df)
    subjects = stats['subjectID'].tolist()
    n_subj = len(subjects)

    n_train = int(round(ratios[0] * n_subj))
    n_val = int(round(ratios[1] * n_subj))
    n_test = n_subj - n_train - n_val

    best = None
    best_info = None
    best_split = None

    for _ in range(n_iter):
        perm = rng.permutation(subjects)
        train_subj = perm[:n_train].tolist()
        val_subj = perm[n_train:n_train+n_val].tolist()
        test_subj = perm[n_train+n_val:].tolist()

        score, info = evaluate_split(stats, train_subj, val_subj, test_subj)
        if (best is None) or (score < best):
            best = score
            best_info = info
            best_split = (train_subj, val_subj, test_subj)

    return best_split, best_info, best

# Exemple
df_q = pd.read_csv('sensor_data_quaternions.csv')
(best_train, best_val, best_test), info, score = random_subject_split_search(df_q, n_iter=30000, seed=42)

print("Best score:", round(score, 4))
print("Train subjects:", sorted(best_train), "| FoG rate:", round(info['train_rate'], 4), "| size:", round(info['train_size'], 3))
print("Val subjects:", sorted(best_val), "| FoG rate:", round(info['val_rate'], 4), "| size:", round(info['val_size'], 3))
print("Test subjects:", sorted(best_test), "| FoG rate:", round(info['test_rate'], 4), "| size:", round(info['test_size'], 3))
print("Global FoG rate:", round(info['global_rate'], 4))

Best score: 0.0108
Train subjects: [2, 3, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 21] | FoG rate: 0.1942 | size: 0.603
Val subjects: [1, 5, 6, 7] | FoG rate: 0.195 | size: 0.192
Test subjects: [4, 8, 19, 20, 22] | FoG rate: 0.1971 | size: 0.205
Global FoG rate: 0.1949


In [161]:
train_subjects = [2, 3, 10, 12, 13, 14, 16, 17, 18, 21]
val_subjects = [1, 5, 6, 7, 11, 15, 20]
test_subjects = [4, 8, 9, 19, 22] 

In [159]:
df_quat = pd.read_csv('sensor_data_complete.csv')

In [160]:

# Analyze the distribution of activities per subject
activity_distribution = df_quat.groupby('subjectID')['activity'].unique().apply(list)
print("Distribution of activities per subject:")
print(activity_distribution)

# Check which subjects perform activities 1 and 3
subjects_with_activity_1 = df_quat[df_quat['activity'] == 1]['subjectID'].unique()
subjects_with_activity_3 = df_quat[df_quat['activity'] == 3]['subjectID'].unique()

print(f"\nSubjects who perform activity 1: {subjects_with_activity_1}")
print(f"Subjects who perform activity 3: {subjects_with_activity_3}")

# Cross-reference with train/val/test splits
print(f"\nTrain subjects: {train_subjects}")
print(f"Val subjects: {val_subjects}")
print(f"Test subjects: {test_subjects}")

subjects_in_train_with_activity_1 = [s for s in subjects_with_activity_1 if s in train_subjects]
subjects_in_train_with_activity_3 = [s for s in subjects_with_activity_3 if s in train_subjects]

subjects_in_val_with_activity_1 = [s for s in subjects_with_activity_1 if s in val_subjects]
subjects_in_val_with_activity_3 = [s for s in subjects_with_activity_3 if s in val_subjects]

subjects_in_test_with_activity_1 = [s for s in subjects_with_activity_1 if s in test_subjects]
subjects_in_test_with_activity_3 = [s for s in subjects_with_activity_3 if s in test_subjects]


print(f"\nSubjects in TRAINING set performing activity 1: {subjects_in_train_with_activity_1}")
print(f"Subjects in TRAINING set performing activity 3: {subjects_in_train_with_activity_3}")

print(f"\nSubjects in VALIDATION set performing activity 1: {subjects_in_val_with_activity_1}")
print(f"Subjects in VALIDATION set performing activity 3: {subjects_in_val_with_activity_3}")

print(f"\nSubjects in TEST set performing activity 1: {subjects_in_test_with_activity_1}")
print(f"Subjects in TEST set performing activity 3: {subjects_in_test_with_activity_3}")


Distribution of activities per subject:
subjectID
1        [2, 4, 1, 7, 5, 3, 6]
2        [2, 4, 3, 1, 6, 5, 7]
3        [2, 4, 3, 1, 6, 7, 5]
4        [4, 3, 1, 6, 5, 2, 7]
5        [2, 4, 1, 6, 7, 5, 3]
6        [2, 4, 1, 6, 7, 5, 3]
7        [2, 4, 1, 6, 7, 5, 3]
8        [2, 4, 1, 6, 5, 7, 3]
9     [2, 4, 1, 6, 7, 5, 3, 0]
10    [3, 5, 2, 4, 1, 7, 6, 0]
11                [3, 1, 7, 6]
12       [2, 4, 3, 1, 7, 6, 0]
13       [2, 4, 1, 6, 5, 3, 7]
14    [2, 4, 1, 7, 5, 3, 6, 0]
15          [2, 4, 3, 1, 7, 6]
16    [2, 4, 1, 7, 6, 5, 3, 0]
17          [1, 7, 5, 2, 3, 6]
18       [2, 4, 1, 7, 6, 5, 3]
19                [3, 1, 7, 6]
20       [2, 4, 1, 6, 7, 5, 3]
21    [2, 4, 1, 7, 6, 5, 3, 0]
22                [3, 1, 7, 6]
Name: activity, dtype: object

Subjects who perform activity 1: [ 1  2  3  4  5  6  7  8 10 11 12 13 14 15 16 17 18 19 20 21 22  9]
Subjects who perform activity 3: [ 1  2  3  4  5  6  7  8 10 11 12 13 14 15 16 17 18 19 20 21 22  9]

Train subjects: [2, 3, 10, 12, 13,

In [ ]:
def segment_data(df, subjects, w=2, o=0.75, fs=60):
    win_size = int(w * fs)
    step = int(win_size * (1 - o))
    
    # Order matters for the Multi-Branch CNN: 8 Ankle channels first, 4 Back channels last
    cols = [
        'ankleL_q0', 'ankleL_q1', 'ankleL_q2', 'ankleL_q3',
        'ankleR_q0', 'ankleR_q1', 'ankleR_q2', 'ankleR_q3',
        'back_q0', 'back_q1', 'back_q2', 'back_q3'
    ]
    
    X, y = [], []
    
    for sub in subjects:
        sub_df = df[df['subjectID'] == sub].reset_index(drop=True)
        data = sub_df[cols].values
        labels = sub_df['fog'].values
        
        for i in range(0, len(data) - win_size, step):
            window = data[i : i + win_size]
            # Subtract the first sample of the window from the whole window
            window_rel = window - window[0]
            
            X.append(window_rel)
            # Use the mode (most frequent) label for the window
            y.append(pd.Series(labels[i : i + win_size]).mode()[0])
            
    return np.array(X), np.array(y)

# Load your generated quaternions file
df_quat = pd.read_csv('sensor_data_quaternions.csv')

print("Segmenting datasets...")
X_train, y_train = segment_data(df_quat, train_subjects)
X_val, y_val     = segment_data(df_quat, val_subjects)
X_test, y_test   = segment_data(df_quat, test_subjects)

Segmenting datasets...


In [163]:
# Après avoir segment_data() appliqué à train/val/test
train_stats = pd.Series(y_train).value_counts()  # comptage fenêtres
val_stats = pd.Series(y_val).value_counts()
test_stats = pd.Series(y_test).value_counts()

print(f"Train window-level: {100*train_stats[1]/(train_stats[0]+train_stats[1]):.1f}% FoG")
print(f"Val window-level: {100*val_stats[1]/(val_stats[0]+val_stats[1]):.1f}% FoG")
print(f"Test window-level: {100*test_stats[1]/(test_stats[0]+test_stats[1]):.1f}% FoG")

Train window-level: 19.4% FoG
Val window-level: 19.6% FoG
Test window-level: 19.4% FoG


In [164]:
import torch
from torch.utils.data import Dataset, DataLoader

class FogDataset(Dataset):
    def __init__(self, X, y):
        # Convert to float32 and move to tensors
        # X shape from segmentation: (Samples, Time, Channels) -> e.g., (N, 120, 12)
        # PyTorch Conv1d expects: (Samples, Channels, Time) -> e.g., (N, 12, 120)
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- Usage ---
# Assuming you already have X_train, y_train, etc., from your segmentation script
train_dataset = FogDataset(X_train, y_train)
val_dataset = FogDataset(X_val, y_val)
test_dataset = FogDataset(X_test, y_test)

# DataLoader handles batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [140]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for the loss function
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# Convert to a tensor for PyTorch (pos_weight for BCEWithLogitsLoss)
pos_weight = torch.tensor([weights[1] / weights[0]], dtype=torch.float32).to(device)

print(f"Class weights: {weights} | Pos_weight: {pos_weight.item():.2f}")

Class weights: [0.62028903 2.57832744] | Pos_weight: 4.16


## Model architecture and training

In [165]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiBranchCNN(nn.Module):
    def __init__(self):
        super(MultiBranchCNN, self).__init__()
        
        # Branch 1: Ankles (8 channels: Left q0-q3 + Right q0-q3)
        self.ankle_branch = nn.Sequential(
            nn.Conv1d(8, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Reduces to a single vector
        )
        
        # Branch 2: Back (4 channels: Back q0-q3)
        self.back_branch = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        
        # Fusion Layer: Combines 128 (Ankles) + 64 (Back) = 192 features
        self.classifier = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x is assumed to be (Batch, Channels, Time) -> (Batch, 12, 120)
        ankles = x[:, 0:8, :] # First 8 channels
        back = x[:, 8:12, :]  # Last 4 channels
        
        feat_ankles = self.ankle_branch(ankles).view(x.size(0), -1)
        feat_back = self.back_branch(back).view(x.size(0), -1)
        
        # Concatenate branches
        combined = torch.cat((feat_ankles, feat_back), dim=1)
        return self.classifier(combined)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = MultiBranchCNN().to(device)
print(f"Model initialized on: {device}")

Model initialized on: mps


In [166]:
import numpy as np

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    # Setup Optimizer and Loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    #criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    criterion = nn.BCELoss()
    #criterion = FocalLoss(alpha=0.75, gamma=2.0)

    # Scheduler: Reduces LR if validation loss doesn't improve for 5 epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_losses = []
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        # --- VALIDATION PHASE ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze(-1)
                v_loss = criterion(outputs, labels)
                val_losses.append(v_loss.item())
        
        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        
        # Update Scheduler
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_multibranch_model_new_split_2.pth')
            print("--> Model Saved!")

# --- START TRAINING ---
train_model(model, train_loader, val_loader)

Epoch 1/50 | Train Loss: 0.4209 | Val Loss: 0.3884 | LR: 0.001000
--> Model Saved!
Epoch 2/50 | Train Loss: 0.3329 | Val Loss: 0.3687 | LR: 0.001000
--> Model Saved!
Epoch 3/50 | Train Loss: 0.3070 | Val Loss: 0.3907 | LR: 0.001000
Epoch 4/50 | Train Loss: 0.3067 | Val Loss: 0.3715 | LR: 0.001000
Epoch 5/50 | Train Loss: 0.2948 | Val Loss: 0.3713 | LR: 0.001000
Epoch 6/50 | Train Loss: 0.2942 | Val Loss: 0.3761 | LR: 0.001000
Epoch 7/50 | Train Loss: 0.2842 | Val Loss: 0.3583 | LR: 0.001000
--> Model Saved!
Epoch 8/50 | Train Loss: 0.2771 | Val Loss: 0.3555 | LR: 0.001000
--> Model Saved!
Epoch 9/50 | Train Loss: 0.2680 | Val Loss: 0.3605 | LR: 0.001000
Epoch 10/50 | Train Loss: 0.2638 | Val Loss: 0.3497 | LR: 0.001000
--> Model Saved!
Epoch 11/50 | Train Loss: 0.2516 | Val Loss: 0.3562 | LR: 0.001000
Epoch 12/50 | Train Loss: 0.2561 | Val Loss: 0.3628 | LR: 0.001000
Epoch 13/50 | Train Loss: 0.2479 | Val Loss: 0.3531 | LR: 0.001000
Epoch 14/50 | Train Loss: 0.2492 | Val Loss: 0.3392 |

## Grouped K-Fold Cross-Validation

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


# Reload and re-segment all data (train + val combined)
train_val_subjects = train_subjects + val_subjects
X_train_val, y_train_val = segment_data(df_quat, train_val_subjects, w=2, o=0.75, fs=60)

print(f"Total windows (train+val): {len(X_train_val)}")
print(f"Train+Val subjects: {sorted(train_val_subjects)}")

groups = []
for subject in train_val_subjects:
    X_subj, y_subj = segment_data(df_quat, [subject], w=2, o=0.75, fs=60)
    groups.extend([subject] * len(X_subj))
groups = np.array(groups)

print(f"Group array shape: {groups.shape}")
assert len(groups) == len(X_train_val), "Mismatch in window/group counts!"

# Initialize GroupKFold: by default, uses n_splits=5
n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

# Storage for CV results
cv_results = {
    'fold': [],
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': [],
    'train_loss': [],
    'val_loss': []
}

print(f"\n{'='*70}")
print(f"  GROUPED K-FOLD CROSS-VALIDATION ({n_splits} folds)")
print(f"{'='*70}\n")

fold_idx = 0
for train_idx, val_idx in gkf.split(X_train_val, y_train_val, groups=groups):
    fold_idx += 1
    print(f"\n{'─'*70}")
    print(f"FOLD {fold_idx}/{n_splits}")
    print(f"{'─'*70}")
    
    # Split data
    X_fold_train, X_fold_val = X_train_val[train_idx], X_train_val[val_idx]
    y_fold_train, y_fold_val = y_train_val[train_idx], y_train_val[val_idx]
    
    # Get train/val subjects for this fold
    fold_train_subjects = np.unique(groups[train_idx])
    fold_val_subjects = np.unique(groups[val_idx])
    
    print(f"Train subjects: {sorted(fold_train_subjects.astype(int).tolist())}")
    print(f"  → {len(X_fold_train)} windows, {100*y_fold_train.mean():.1f}% FoG")
    print(f"Val subjects: {sorted(fold_val_subjects.astype(int).tolist())}")
    print(f"  → {len(X_fold_val)} windows, {100*y_fold_val.mean():.1f}% FoG")
    
    # Create fresh datasets and loaders for this fold
    fold_train_dataset = FogDataset(X_fold_train, y_fold_train)
    fold_val_dataset = FogDataset(X_fold_val, y_fold_val)
    
    fold_train_loader = torch.utils.data.DataLoader(fold_train_dataset, batch_size=64, shuffle=True)
    fold_val_loader = torch.utils.data.DataLoader(fold_val_dataset, batch_size=64, shuffle=False)
    
    # Reinitialize model for this fold
    fold_model = MultiBranchCNN().to(device)
    
    # Training
    optimizer = torch.optim.Adam(fold_model.parameters(), lr=0.001, weight_decay=1e-3)
    criterion = nn.BCELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
    
    best_fold_val_loss = float('inf')
    fold_train_losses = []
    fold_val_losses = []
    
    epochs = 30  # Fewer epochs for CV to save time
    for epoch in range(epochs):
        fold_model.train()
        train_losses = []
        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = fold_model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        fold_model.eval()
        val_losses = []
        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = fold_model(inputs).squeeze()
                v_loss = criterion(outputs, labels)
                val_losses.append(v_loss.item())
        
        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_fold_val_loss:
            best_fold_val_loss = avg_val_loss
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} | Loss: {avg_train_loss:.4f} / {avg_val_loss:.4f}")
    
    # Predictions on fold validation set
    fold_model.eval()
    fold_val_probs = []
    fold_val_preds = []
    with torch.no_grad():
        for inputs, labels in fold_val_loader:
            inputs = inputs.to(device)
            outputs = fold_model(inputs).squeeze()
            if outputs.dim() == 0:
                outputs = outputs.unsqueeze(0)
            probs = outputs
            preds = (probs > 0.3).int()
            fold_val_probs.extend(probs.cpu().numpy().tolist())
            fold_val_preds.extend(preds.cpu().numpy().tolist())
    
    fold_val_preds = np.array(fold_val_preds, dtype=int)
    
    # Metrics
    fold_acc = accuracy_score(y_fold_val, fold_val_preds)
    fold_prec = precision_score(y_fold_val, fold_val_preds, zero_division=0)
    fold_rec = recall_score(y_fold_val, fold_val_preds, zero_division=0)
    fold_f1 = f1_score(y_fold_val, fold_val_preds, zero_division=0)
    
    cv_results['fold'].append(fold_idx)
    cv_results['accuracy'].append(fold_acc)
    cv_results['precision'].append(fold_prec)
    cv_results['recall'].append(fold_rec)
    cv_results['f1'].append(fold_f1)
    cv_results['train_loss'].append(np.mean(fold_train_losses))
    cv_results['val_loss'].append(best_fold_val_loss)
    
    print(f"\nFold {fold_idx} Results:")
    print(f"  Accuracy:  {fold_acc:.4f}")
    print(f"  Precision: {fold_prec:.4f}")
    print(f"  Recall:    {fold_rec:.4f}")
    print(f"  F1-Score:  {fold_f1:.4f}")

# Summarize CV results
cv_df = pd.DataFrame(cv_results)

print(f"\n{'='*70}")
print(f"  CV SUMMARY ({n_splits} folds)")
print(f"{'='*70}\n")
print(cv_df.to_string(index=False))

print(f"\n{'─'*70}")
print(f"CROSS-VALIDATION AVERAGES:")
print(f"{'─'*70}")
print(f"Accuracy:  {cv_df['accuracy'].mean():.4f} (±{cv_df['accuracy'].std():.4f})")
print(f"Precision: {cv_df['precision'].mean():.4f} (±{cv_df['precision'].std():.4f})")
print(f"Recall:    {cv_df['recall'].mean():.4f} (±{cv_df['recall'].std():.4f})")
print(f"F1-Score:  {cv_df['f1'].mean():.4f} (±{cv_df['f1'].std():.4f})")
print(f"\nConclusion: Model generalizes across {n_splits} folds of train+val subjects.")

Total windows (train+val): 7599
Train+Val subjects: [1, 2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21]
Group array shape: (7599,)

  GROUPED K-FOLD CROSS-VALIDATION (5 folds)


──────────────────────────────────────────────────────────────────────
FOLD 1/5
──────────────────────────────────────────────────────────────────────
Train subjects: [1, 3, 6, 7, 10, 11, 12, 13, 14, 15, 17, 18, 20, 21]
  → 5908 windows, 14.6% FoG
Val subjects: [2, 5, 16]
  → 1691 windows, 36.7% FoG
  Epoch 10/30 | Loss: 0.2366 / 0.5456
  Epoch 20/30 | Loss: 0.2115 / 0.5016
  Epoch 30/30 | Loss: 0.2071 / 0.4976

Fold 1 Results:
  Accuracy:  0.7830
  Precision: 0.8504
  Recall:    0.4952
  F1-Score:  0.6259

──────────────────────────────────────────────────────────────────────
FOLD 2/5
──────────────────────────────────────────────────────────────────────
Train subjects: [2, 3, 5, 7, 10, 11, 12, 13, 14, 15, 16, 17, 20]
  → 6101 windows, 18.3% FoG
Val subjects: [1, 6, 18, 21]
  → 1498 windows, 24.2% F

In [ ]:
print(f"\n{'='*70}")
print(f"  FINAL TRAINING ON FULL TRAIN+VAL SET")
print(f"{'='*70}\n")

# Reinit model
final_model = MultiBranchCNN().to(device)

# Use full train+val data
final_train_dataset = FogDataset(X_train_val, y_train_val)
final_train_loader = torch.utils.data.DataLoader(final_train_dataset, batch_size=64, shuffle=True)

# Training setup
optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001, weight_decay=1e-3)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

best_final_loss = float('inf')
final_epochs = 50

print(f"Training on {len(X_train_val)} windows from {len(train_val_subjects)} subjects")
print(f"FoG prevalence in train+val: {100*y_train_val.mean():.1f}%\n")

for epoch in range(final_epochs):
    final_model.train()
    train_losses = []
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    avg_train_loss = np.mean(train_losses)
    scheduler.step(avg_train_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{final_epochs} | Loss: {avg_train_loss:.4f}")
    
    if avg_train_loss < best_final_loss:
        best_final_loss = avg_train_loss
        torch.save(final_model.state_dict(), 'final_model_full_trainval.pth')

print(f"\n✓ Final model saved!")

print(f"\n{'='*70}")
print(f"  FINAL EVALUATION ON TEST SET")
print(f"{'='*70}\n")

final_model.eval()
test_probs = []
test_preds = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = final_model(inputs).squeeze()
        if outputs.dim() == 0:
            outputs = outputs.unsqueeze(0)
        probs = outputs
        preds = (probs > 0.3).int()
        test_probs.extend(probs.cpu().numpy().tolist())
        test_preds.extend(preds.cpu().numpy().tolist())

test_preds = np.array(test_preds, dtype=int)
test_probs = np.array(test_probs)

# Compute metrics
test_acc = accuracy_score(y_test, test_preds)
test_prec = precision_score(y_test, test_preds, zero_division=0)
test_rec = recall_score(y_test, test_preds, zero_division=0)
test_f1 = f1_score(y_test, test_preds, zero_division=0)
test_cm = confusion_matrix(y_test, test_preds, labels=[0, 1])
test_auc = roc_auc_score(y_test, test_probs)

print(f"Test set: {len(y_test)} windows from subjects {sorted(best_test)}")
print(f"FoG prevalence in test: {100*y_test.mean():.1f}%\n")

print(f"{'─'*70}")
print(f"TEST RESULTS:")
print(f"{'─'*70}")
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"AUC-ROC:   {test_auc:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TN={test_cm[0,0]}, FP={test_cm[0,1]}")
print(f"  FN={test_cm[1,0]}, TP={test_cm[1,1]}")

print(f"\n{'='*70}")
print(f"  CV vs FINAL TEST COMPARISON")
print(f"{'='*70}\n")

comparison_data = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
    'CV Mean': [
        cv_df['accuracy'].mean(),
        cv_df['precision'].mean(),
        cv_df['recall'].mean(),
        cv_df['f1'].mean(),
        np.nan  # AUC not in CV results
    ],
    'CV Std': [
        cv_df['accuracy'].std(),
        cv_df['precision'].std(),
        cv_df['recall'].std(),
        cv_df['f1'].std(),
        np.nan
    ],
    'Test': [test_acc, test_prec, test_rec, test_f1, test_auc],
    'Difference': [
        test_acc - cv_df['accuracy'].mean(),
        test_prec - cv_df['precision'].mean(),
        test_rec - cv_df['recall'].mean(),
        test_f1 - cv_df['f1'].mean(),
        np.nan
    ]
})

print(comparison_data.to_string(index=False))

print(f"\n{'─'*70}")
print(f"INTERPRETATION:")
print(f"{'─'*70}")

diff_f1 = test_f1 - cv_df['f1'].mean()
if abs(diff_f1) < cv_df['f1'].std():
    print(f"✓ Test performance is CONSISTENT with CV (within 1 std)")
    print(f"  → Model generalizes well to hold-out test set")
elif diff_f1 < 0:
    print(f"⚠ Test F1 is LOWER than CV mean")
    print(f"  → Possible overfitting to train+val")
else:
    print(f"⚠ Test F1 is HIGHER than CV mean (unusual)")
    print(f"  → Possible lucky test set or CV underfitting")

print(f"\nFinal Test F1-Score: {test_f1:.4f}")
print(f"CV Average F1-Score: {cv_df['f1'].mean():.4f} (±{cv_df['f1'].std():.4f})")



  FINAL TRAINING ON FULL TRAIN+VAL SET

Training on 7676 windows from 17 subjects
FoG prevalence in train+val: 19.4%

Epoch 10/50 | Loss: 0.2623
Epoch 20/50 | Loss: 0.2443
Epoch 30/50 | Loss: 0.2417
Epoch 40/50 | Loss: 0.2451
Epoch 50/50 | Loss: 0.2327

✓ Final model saved!

  FINAL EVALUATION ON TEST SET

Test set: 1972 windows from subjects [4, 8, 19, 20, 22]
FoG prevalence in test: 19.7%

──────────────────────────────────────────────────────────────────────
TEST RESULTS:
──────────────────────────────────────────────────────────────────────
Accuracy:  0.8169
Precision: 0.5229
Recall:    0.7938
F1-Score:  0.6305
AUC-ROC:   0.8785

Confusion Matrix:
  TN=1303, FP=281
  FN=80, TP=308

  CV vs FINAL TEST COMPARISON

   Metric  CV Mean   CV Std     Test  Difference
 Accuracy 0.816580 0.074672 0.816937    0.000357
Precision 0.479356 0.240676 0.522920    0.043564
   Recall 0.635032 0.148488 0.793814    0.158783
 F1-Score 0.508319 0.125029 0.630502    0.122183
  AUC-ROC      NaN      NaN 

In [167]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def evaluate_model(model, test_loader, device, threshold=0.7):
    """
    Evaluates the model on the test set and returns true labels, predictions, and probabilities.
    This version is corrected to handle single-item batches.
    """
    model.eval()  # Set model to evaluation mode
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():  # No gradients needed for evaluation
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            
            # Get model output
            outputs = model(inputs).squeeze()
            
            # --- CORRECTION FOR SINGLE-ITEM BATCH ---
            # If the batch has only one sample, 'outputs' becomes a 0-D tensor (scalar).
            # We need to handle it differently from a 1-D tensor (vector).
            if outputs.dim() == 0:
                # It's a single number, so we append it directly
                all_probs.append(outputs.cpu().item())
                pred = (outputs > threshold).float()
                all_preds.append(pred.cpu().item())
            else:
                # It's a vector for a multi-item batch, so we use extend
                all_probs.extend(outputs.cpu().numpy())
                preds = (outputs > threshold).float()
                all_preds.extend(preds.cpu().numpy())
            
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# --- EXECUTION ---
# Load the best weights saved during training
# Make sure the model is already defined in a previous cell
model.load_state_dict(torch.load('best_multibranch_model_new_split_2.pth'))

# Evaluate the model with the corrected function
y_true, y_pred, y_probs = evaluate_model(model, test_loader, device, threshold=0.4)

print("Evaluation completed successfully.")
print(f"y_true length: {len(y_true)}")
print(f"y_pred length: {len(y_pred)}")
print(f"y_probs length: {len(y_probs)}")

Evaluation completed successfully.
y_true length: 2049
y_pred length: 2049
y_probs length: 2049


In [168]:
print("\n" + "="*30)
print("   FINAL TEST RESULTS")
print("="*30)

# 1. Classification Report
print(classification_report(y_true, y_pred, target_names=['Normal', 'FoG']))

# 2. AUC Score (measures how well the model separates the two classes)
auc = roc_auc_score(y_true, y_probs)
print(f"Test AUC-ROC: {auc:.4f}")

# 3. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)


   FINAL TEST RESULTS
              precision    recall  f1-score   support

      Normal       0.91      0.90      0.90      1651
         FoG       0.60      0.63      0.62       398

    accuracy                           0.85      2049
   macro avg       0.76      0.76      0.76      2049
weighted avg       0.85      0.85      0.85      2049

Test AUC-ROC: 0.8936

Confusion Matrix:
[[1486  165]
 [ 148  250]]
